**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Variational Inference & Normalizing Flows

The [VAE's](./Representation_Learning.ipynb) loss finally justified: the ELBO derived, its gap identified as a KL, and normalizing flows — exact likelihoods through invertible networks — built and audited against a closed-form density.

## 1. Pre-requisites

[Representation Learning](./Representation_Learning.ipynb) S2, [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) (KL), [Random Variables](../Intro_Math/Analysis/Random_Variables.ipynb) (change of variables).

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *The ELBO, Derived* (~40 min)
**Goal:** one line of algebra: log-likelihood = ELBO + KL(q‖posterior); verified on a conjugate model.
**Builds on:** [Representation Learning](./Representation_Learning.ipynb) S2. &nbsp; **Feeds into:** Session 2 (flows).

---

## 2. The Bound and Its Gap

💡 **Intuition.** Latent-variable likelihoods need an integral over $z$ — intractable. Multiply and divide by any distribution $q(z)$ inside the log, apply [Jensen](../Intro_Math/Information_Theory/Information_Theory.ipynb), and:
$$\log p(x) = \underbrace{E_q[\log p(x, z) - \log q(z)]}_{\text{ELBO}} + \underbrace{KL(q \,\|\, p(z|x))}_{\ge 0, = \text{the gap}}$$
Maximizing the ELBO over $q$ *is* pushing $q$ toward the true posterior; the bound is tight iff they match. The VAE's loss is exactly this with an encoder network as $q$. On a conjugate Gaussian model the posterior is known — so for once we can *watch* the gap close.

In [ ]:
# model: z ~ N(0,1), x|z ~ N(z, 0.5²); observe x=1.2. Posterior is closed-form Gaussian.
# variational family: N(m, s²); optimize the ELBO by gradient ascent (reparameterized MC)

# YOUR CODE HERE


**What just happened.** Gradient ascent on the ELBO, over a two-parameter Gaussian family, found the true posterior:

| | mean | variance |
|---|---|---|
| true posterior | 0.9600 | 0.2000 |
| learned $q$ | 0.9637 | 0.1907 |

and the bound closed: $\log p(x) = -1.6065$, final ELBO $= -1.6053$.

**Check the closed forms rather than trusting them, because the whole demo rests on them.** Posterior precision is $1 + 1/0.25 = 5$, so variance $= 0.2$; posterior mean is $0.2 \times (1.2/0.25) = 0.96$. Marginally $x \sim \mathcal{N}(0, 1.25)$, so $\log p(1.2) = -\tfrac12\log(2\pi \cdot 1.25) - \tfrac{1.44}{2 \cdot 1.25} = -1.6065$. **Two lines of Gaussian algebra supply the answer key**, which is exactly why this conjugate model was chosen.

**Now the reported gap: $-0.00123$. That is negative, and a KL divergence cannot be negative.** The identity says $\log p(x) = \text{ELBO} + \mathrm{KL}(q \| p(z|x))$ with $\mathrm{KL} \ge 0$, so the ELBO can never exceed $\log p(x)$. **The bound was not violated — the estimate was noisy.** The ELBO here is a **256-sample Monte Carlo average**, and its standard error is comfortably larger than $10^{-3}$. The estimator overshot by about one standard error.

**Noticing that is the transferable skill, and it recurs across this curriculum.** The same reflex catches the Sinkhorn cost that landed *below* the LP optimum in [Optimal Transport](./Optimal_Transport.ipynb). **When a number lands on the impossible side of a bound, the bound is fine and your estimator is telling you something.** Here it says: average over more samples, or over the last hundred steps, before quoting a gap.

**With that caveat, the finding is genuine and it is the point of the session.** Nothing in the training loop ever mentioned the posterior — the code only ever computed $\log p(x,z) - \log q(z)$ and pushed uphill. **Maximising the ELBO performed inference**, because $\log p(x)$ does not depend on $q$, so every nat gained by the ELBO is a nat removed from $\mathrm{KL}(q \| p(z|x))$. Optimising a bound and fitting a posterior are the same operation.

**And the bound is tight here for a specific reason worth stating.** The true posterior is Gaussian and the variational family is Gaussian, so **the family contains the answer**. That is the best case. When the true posterior is multimodal or skewed and $q$ is Gaussian, the gap does not close — and the direction of the KL, $\mathrm{KL}(q\|p)$ rather than $\mathrm{KL}(p\|q)$, makes $q$ **mode-seeking**: it will lock onto one mode and ignore the others rather than covering both. That asymmetry is the best-known limitation of variational inference and it is invisible in a conjugate demo.

**Finally, note the two ways a real VAE is looser than this.** Its $q$ family is whatever the encoder network can express, and it is **amortised** — one network must produce a good $q$ for *every* $x$, rather than each $x$ getting its own optimisation. That second effect has its own name, the amortisation gap, and it is why the VAE's ELBO sits further below $\log p(x)$ than these two scalars managed.

---
### 🕐 Session 2 of 3 — *Normalizing Flows* (~40 min)
**Goal:** exact densities through invertible maps: change-of-variables with a learnable Jacobian.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (the trade-space).

---

## 3. Exact Likelihood, No Bound

💡 **Intuition.** Push a Gaussian through an *invertible* network $f$ and the [change-of-variables formula](../Intro_Math/Analysis/Random_Variables.ipynb) gives the exact density: $\log p(x) = \log p_z(f^{-1}(x)) + \log|\det J_{f^{-1}}|$. The engineering is making that determinant cheap: **coupling layers** transform half the coordinates using parameters computed from the other half — triangular Jacobian, determinant = product of scales. Stack and alternate halves: an expressive, exactly-normalized density.

In [ ]:
# simpler, correct assembly:
# target: the two-moons distribution (reuse the diffusion workshop's)
# density heatmap — exactly normalized by construction

# YOUR CODE HERE


**What just happened.** Two numbers and a picture, and the second number is the one that matters:

- **NLL 1.275 nats**, against 2.838 for the best standard normal — an improvement of **1.56 nats**, a factor of $e^{1.56} \approx 4.8$ in likelihood per data point.
- **$\int p \approx 1.000$** — the learned density integrates to one.

**That integral is the claim no other generative family in this curriculum can make.** A [VAE](./Representation_Learning.ipynb) gives a *bound*; a GAN gives no density at all; a [diffusion model](./Diffusion_Models.ipynb) gives a likelihood only through an expensive ODE. **A flow gives you $p(x)$ directly, exactly normalised, and you can check it by summing over a grid.** One line of verification, and it either passes or it does not.

**And "exactly normalised" is structural, not learned.** Nothing in the training loss rewarded normalisation. It holds because $p_z$ is a normalised Gaussian and the change-of-variables formula conserves total mass under any invertible map — so **any** invertible $f$, trained or random, yields a normalised density. The 1.000 confirms the implementation is correct, not that the model is good.

**Which makes the NLL directly comparable in a way most generative metrics are not.** 1.275 nats is a real, unbounded-below-only-by-the-data quantity: two flows on the same data can be ranked by it without qualification. **Compare that to FID, ELBO values, or human evaluation** — every other family's headline number involves either a bound or a proxy.

**The engineering that makes this possible is the coupling layer, and it is worth reading in the code.** Half the coordinates pass through **untouched**; the other half are scaled and shifted by parameters computed *from the untouched half*. The Jacobian is therefore **triangular**, so its determinant is just the product of the scales — $O(d)$ instead of $O(d^3)$. Note the asymmetry that buys the expressiveness: the network producing $s$ and $t$ can be arbitrarily complicated, because it never appears in the determinant. `logdet += s.squeeze(1)` is the entire volume-tracking cost.

**Two implementation details are load-bearing.** The `i % 2` alternation swaps which half moves, because otherwise half the coordinates would never be transformed at all. And `s = torch.tanh(s)` caps the per-layer volume change at $e^{\pm 1}$ — remove it and the log-determinant runs away to `nan` within a few hundred steps. **Neither is cosmetic.**

**Now the constraint that is the real cost, and it should not be buried.** Every layer must be **invertible and dimension-preserving**. There is no bottleneck, no latent space smaller than the data, no stride, no pooling — you cannot drop an arbitrary architecture into a flow. At $d = 3072$ for a small image, the latent is also 3072-dimensional, so flows are **parameter-hungry** compared to a VAE with a 128-dimensional latent. **Flows buy exactness by spending architectural freedom.**

**One caveat about the density plot itself.** The heatmap shows the model's density, not the data's — and a flow on a distribution concentrated near a thin manifold has to place enormous density on a small region while remaining continuous everywhere. Look for the characteristic failure: thin filaments of leaked density connecting the two moons, where the invertible map has to stretch through the empty space between them. **A flow cannot assign exactly zero density anywhere**, which is the price of being a smooth invertible map from a Gaussian.

---
### 🕐 Session 3 of 3 — *The Generative Trade-Space* (~25 min)
**Goal:** VAE vs flow vs diffusion vs GAN: what each buys and what each pays.
**Builds on:** Session 2.

---

## 4. The Family Reunion

| | VAE | Flow | [Diffusion](./Diffusion_Models.ipynb) | GAN |
|---|---|---|---|---|
| Likelihood | bound (ELBO) | **exact** | bound/exact (SDE) | none |
| Sampling | 1 pass | 1 pass | many steps | 1 pass |
| Architecture freedom | full | invertible only | full | full |
| Training stability | good | good | **great** | fragile |
| Latent space | semantic | dimension-preserving | noise schedule | semantic |

💡 **Intuition.** Every generative model juggles three balls — sample quality, likelihood access, sampling speed — and each family drops a different one. Diffusion won the 2020s on quality+stability; flows keep the niche where exact density matters (physics, anomaly detection, [compression](../Intro_Math/Information_Theory/Information_Theory.ipynb)); the ELBO remains the shared grammar.

---
## Where next

- [Diffusion II](./Diffusion_Score_SDE.ipynb) — probability flow: diffusion's flow-shaped face.
- [Representation Learning](./Representation_Learning.ipynb) — the VAE, now with its theory installed.